# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [70]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [71]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_with_target = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_with_target['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [72]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [73]:
## SVM
svc = SVC(random_state=21,
          kernel='linear',
          gamma='auto',
          class_weight=None)
svc.fit(X_train, y_train)
pred = svc.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.71006
precision is 0.73495
recall is: 0.71006


In [74]:
## Decision Tree
dt = DecisionTreeClassifier(random_state=21,
                            class_weight=None,
                            criterion='gini',
                            max_depth=31)
dt.fit(X_train, y_train)
pred = dt.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.88166
precision is 0.88375
recall is: 0.88166


In [75]:
## Random Forest
rf = RandomForestClassifier(random_state=21,
                            n_estimators=100,
                            max_depth=31, 
                            class_weight=None,
                            criterion='gini')
rf.fit(X_train, y_train)
pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, average='weighted')
recall = recall_score(y_test, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.90828
precision is 0.91059
recall is: 0.90828


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [76]:
voting_clf = VotingClassifier(
    estimators=[
    ('svc', svc),
    ('dt', dt),
    ('rf', rf)],
    voting='hard'
)

In [77]:
voting_clf.fit(X_train, y_train)
pred = voting_clf.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.86667
precision is 0.86671
recall is: 0.86667


In [78]:
# Список вариантов весов
weights_variants = [
    [1, 1, 1],    # Равные веса
    [2, 1, 3],    # Разные веса
    [3, 1, 2],    # Другое распределение
    [1, 3, 2]     # Еще вариант
]

best_accuracy = 0
best_precision = 0
best_voting_clf = None

# Перебор вариантов весов
for weights in weights_variants:
    voting_clf = VotingClassifier(
        estimators=[
            ('svc', svc),
            ('dt', dt),
            ('rf', rf)
        ],
        voting='hard',
        weights=weights
    )
    
    # Обучение
    voting_clf.fit(X_train, y_train)
    
    # Предсказание на validation
    pred = voting_clf.predict(X_valid)
    
    # Вычисление метрик
    accuracy = accuracy_score(y_valid, pred)
    precision = precision_score(y_valid, pred, average='weighted')
    recall = recall_score(y_valid, pred, average='weighted')
    
    print(f"Веса {weights}:")
    print(f'Accuracy: {accuracy:.5f}')
    print(f'Precision: {precision:.5f}')
    print(f'Recall: {recall:.5f}\n')
    
    # Выбор лучшей модели
    if (accuracy > best_accuracy) or \
       (accuracy == best_accuracy and precision > best_precision):
        best_accuracy = accuracy
        best_precision = precision
        best_voting_clf = voting_clf

Веса [1, 1, 1]:
Accuracy: 0.86667
Precision: 0.86671
Recall: 0.86667

Веса [2, 1, 3]:
Accuracy: 0.88148
Precision: 0.88172
Recall: 0.88148

Веса [3, 1, 2]:
Accuracy: 0.74074
Precision: 0.75003
Recall: 0.74074

Веса [1, 3, 2]:
Accuracy: 0.86296
Precision: 0.86263
Recall: 0.86296



In [79]:
# Оценка на test set
test_pred = best_voting_clf.predict(X_test)
    
test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average='weighted')
test_recall = recall_score(y_test, test_pred, average='weighted')

print("Результаты на test set:")
print(f'Accuracy: {test_accuracy:.5f}')
print(f'Precision: {test_precision:.5f}')
print(f'Recall: {test_recall:.5f}')

Результаты на test set:
Accuracy: 0.90828
Precision: 0.91059
Recall: 0.90828


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [80]:
bagging_clf = BaggingClassifier(
    estimator=svc,
    random_state=21,
    bootstrap=True # с возвращением
)

bagging_clf.fit(X_train, y_train)
pred = bagging_clf.predict(X_valid)

accuracy = accuracy_score(y_valid, pred)
precision = precision_score(y_valid, pred, average='weighted')
recall = recall_score(y_valid, pred, average='weighted')

print(f'accuracy is {accuracy:.5}')
print(f'precision is {precision:.5}')
print(f'recall is: {recall:.5}')

accuracy is 0.65185
precision is 0.63628
recall is: 0.65185


In [81]:
# Список параметров для экспериментов
params_list = [
    {'n_estimators': 10, 'max_samples': 0.6},
    {'n_estimators': 20, 'max_samples': 0.7},
    {'n_estimators': 15, 'max_samples': 0.9}
]

best_accuracy = 0
best_precision = 0
best_bagging_clf = None

# Перебор параметров
for params in params_list:
    bagging_clf = BaggingClassifier(
        estimator=svc,
        random_state=21,
        bootstrap=True,
        n_estimators=params['n_estimators'],
        max_samples=params['max_samples']
    )
    
    # Обучение
    bagging_clf.fit(X_train, y_train)
    
    # Предсказание на validation
    pred = bagging_clf.predict(X_valid)
    
    # Вычисление метрик
    accuracy = accuracy_score(y_valid, pred)
    precision = precision_score(y_valid, pred, average='weighted')
    recall = recall_score(y_valid, pred, average='weighted')
    
    print(f"Параметры {params}:")
    print(f'Accuracy: {accuracy:.5f}')
    print(f'Precision: {precision:.5f}')
    print(f'Recall: {recall:.5f}\n')
    
    # Выбор лучшей модели
    if (accuracy > best_accuracy) or \
       (accuracy == best_accuracy and precision > best_precision):
        best_accuracy = accuracy
        best_precision = precision
        best_bagging_clf = bagging_clf

Параметры {'n_estimators': 10, 'max_samples': 0.6}:
Accuracy: 0.65185
Precision: 0.66004
Recall: 0.65185

Параметры {'n_estimators': 20, 'max_samples': 0.7}:
Accuracy: 0.62963
Precision: 0.63195
Recall: 0.62963

Параметры {'n_estimators': 15, 'max_samples': 0.9}:
Accuracy: 0.65556
Precision: 0.64833
Recall: 0.65556



In [82]:
## Оценка на test set
test_pred = best_bagging_clf.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, average='weighted')
test_recall = recall_score(y_test, test_pred, average='weighted')

print(f'Accuracy: {test_accuracy:.5f}')
print(f'Precision: {test_precision:.5f}')
print(f'Recall: {test_recall:.5f}')


Accuracy: 0.71598
Precision: 0.73052
Recall: 0.71598


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [83]:
# Список параметров для экспериментов
n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_options = [True, False]

best_accuracy = 0
best_precision = 0
best_stacking_clf = None

# Перебор параметров
for n_splits in n_splits_list:
    for passthrough in passthrough_options:
        # Создаем генератор кросс-валидации
        cv = StratifiedKFold(
            n_splits=n_splits, 
            shuffle=True, 
            random_state=21
        )
        
        # Создаем StackingClassifier
        stacking_clf = StackingClassifier(
            estimators=[
                ('svc', svc),
                ('dt', dt),
                ('rf', rf)
            ],
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=cv,
            passthrough=passthrough
        )
        
        # Обучение
        stacking_clf.fit(X_train, y_train)
        
        # Предсказание на validation
        pred = stacking_clf.predict(X_valid)
        
        # Вычисление метрик
        accuracy = accuracy_score(y_valid, pred)
        precision = precision_score(y_valid, pred, average='weighted')
        recall = recall_score(y_valid, pred, average='weighted')
        
        print(f"Параметры (n_splits={n_splits}, passthrough={passthrough}):")
        print(f'Accuracy: {accuracy:.5f}')
        print(f'Precision: {precision:.5f}')
        print(f'Recall: {recall:.5f}\n')
        
        # Выбор лучшей модели
        if (accuracy > best_accuracy) or \
           (accuracy == best_accuracy and precision > best_precision):
            best_accuracy = accuracy
            best_precision = precision
            best_stacking_clf = stacking_clf

Параметры (n_splits=2, passthrough=True):
Accuracy: 0.89259
Precision: 0.89351
Recall: 0.89259

Параметры (n_splits=2, passthrough=False):
Accuracy: 0.88889
Precision: 0.88927
Recall: 0.88889

Параметры (n_splits=3, passthrough=True):
Accuracy: 0.89259
Precision: 0.89199
Recall: 0.89259

Параметры (n_splits=3, passthrough=False):
Accuracy: 0.89259
Precision: 0.89329
Recall: 0.89259

Параметры (n_splits=4, passthrough=True):
Accuracy: 0.90000
Precision: 0.89816
Recall: 0.90000

Параметры (n_splits=4, passthrough=False):
Accuracy: 0.89630
Precision: 0.89552
Recall: 0.89630

Параметры (n_splits=5, passthrough=True):
Accuracy: 0.89259
Precision: 0.89181
Recall: 0.89259

Параметры (n_splits=5, passthrough=False):
Accuracy: 0.89259
Precision: 0.89317
Recall: 0.89259

Параметры (n_splits=6, passthrough=True):
Accuracy: 0.88889
Precision: 0.88841
Recall: 0.88889

Параметры (n_splits=6, passthrough=False):
Accuracy: 0.88889
Precision: 0.88835
Recall: 0.88889

Параметры (n_splits=7, passthrough=

In [84]:
# Оценка на test set
if best_stacking_clf is not None:
    test_pred = best_stacking_clf.predict(X_test)
    
    test_accuracy = accuracy_score(y_test, test_pred)
    test_precision = precision_score(y_test, test_pred, average='weighted')
    test_recall = recall_score(y_test, test_pred, average='weighted')
    
    print("Результаты на test set:")
    print(f'Accuracy: {test_accuracy:.5f}')
    print(f'Precision: {test_precision:.5f}')
    print(f'Recall: {test_recall:.5f}')

Результаты на test set:
Accuracy: 0.89941
Precision: 0.90270
Recall: 0.89941


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [85]:
prediction = best_voting_clf.predict(X_test)

In [86]:
X_test

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
1087,67,17,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
16,1,13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
563,14,10,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1381,20,15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1199,9,13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1411,156,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1079,59,17,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1222,22,17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1064,50,16,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [87]:
results = pd.DataFrame()
results['y_test'] = y_test
results['prediction'] = prediction

In [88]:
results['is_error'] = (results['y_test'] != results['prediction']).astype(int)

In [89]:
## weekdays
results.groupby('y_test').size().sort_values(ascending=False)

y_test
3    80
6    71
1    55
5    54
2    30
0    27
4    21
dtype: int64

In [90]:
## labnames
# Для каждого labname найдем количество ошибок
labname_columns = [col for col in X_test.columns if col.startswith('labname_')]

# Словарь для хранения ошибок
labname_errors = {}

for col in labname_columns:
    # Находим строки с конкретным labname
    subset = results[X_test[col] == 1.0]
    
    # Считаем количество ошибок
    errors_count = subset['is_error'].sum()
    
    labname_errors[col] = errors_count

# Сортируем по количеству ошибок в убывающем порядке
sorted_labname_errors = sorted(labname_errors.items(), key=lambda x: x[1], reverse=True)

# Вывод результатов
for labname, errors in sorted_labname_errors:
    print(f"{labname}: {errors} errors")


labname_project1: 13 errors
labname_laba04: 7 errors
labname_laba04s: 4 errors
labname_code_rvw: 1 errors
labname_lab03: 1 errors
labname_lab03s: 1 errors
labname_lab05s: 1 errors
labname_laba05: 1 errors
labname_laba06: 1 errors
labname_laba06s: 1 errors
labname_lab02: 0 errors


In [91]:
## users
# Найдем столбцы пользователей
uid_columns = [col for col in X_test.columns if col.startswith('uid_user_')]

# Словарь для хранения ошибок
user_errors = {}

for col in uid_columns:
    # Находим строки с конкретным пользователем
    subset = results[X_test[col] == 1.0]
    
    # Считаем количество ошибок
    errors_count = subset['is_error'].sum()
    
    user_errors[col] = errors_count

# Сортируем по количеству ошибок в убывающем порядке
sorted_user_errors = sorted(user_errors.items(), key=lambda x: x[1], reverse=True)

# Вывод результатов
for username, errors in sorted_user_errors[:10]:  # Топ 10 пользователей
    print(f"{username}: {errors} errors")


uid_user_14: 3 errors
uid_user_19: 3 errors
uid_user_2: 3 errors
uid_user_25: 3 errors
uid_user_10: 2 errors
uid_user_16: 2 errors
uid_user_3: 2 errors
uid_user_31: 2 errors
uid_user_4: 2 errors
uid_user_6: 2 errors


My model makes the most errors for the day number 3 (Wednesday), for labname_project1, for uid_user_14, uid_user_19, uid_user_2, uid_user_25.